In [31]:
# getting transcript

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled


In [32]:
video_id="vWeFTt3pMyc"
try:
    ytt_api = YouTubeTranscriptApi()
    transcript_list=ytt_api.fetch(video_id=video_id)
    
    transcript=" ".join(snippet.text for snippet in transcript_list)
    print(transcript)
    
except TranscriptsDisabled:
    print("No Transcript Available of the Vedio")       
     

Big news coming in right [music] now. It seems two US nationals were detained by Srinagar Airport security after a Garmin satellite phone was found from their luggage during a security check. [music] Remember, it's illegal in India to carry a satphone. in the flight. After initial questioning, they have been handed over to police for further investigation. Now, the individual from whose luggage satellite phone was recovered has been [music] identified as Jeffrey Scott from Montana in USA. [music] The use of satellite phones is prohibited in India without government permission. Previously also US uh nationals have been detained in various [music] Indian airports for carrying such satellite phones. And it is clearly mentioned >> [music] >> in most advisories that satellite phone should not be carried in flights in India. [music] Joining us live now is India Today's Mufti Farid. Mufti, is this uh being investigated as a matter of mere confusion because uh many people are not aware of this

In [33]:
# Splitter

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [34]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=125,
    chunk_overlap=40
)

In [35]:
text_chunks=splitter.split_text(transcript)

print(len(text_chunks))

35


In [36]:
print(text_chunks[0])

Big news coming in right [music] now. It seems two US nationals were detained by Srinagar Airport security after a Garmin


In [37]:
# Embeddings

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

In [38]:
load_dotenv()

True

In [39]:
embeddings=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [40]:
# Vector Store
from langchain_chroma import Chroma

In [41]:
vector_store=Chroma(
    embedding_function=embeddings,
    collection_name="ytt",
    persist_directory="ytt"
)

In [42]:
vector_store.add_texts(text_chunks)

['e073ffd9-93f5-41aa-8fde-29cf2083495c',
 'fafb7d7d-4455-4772-ba4c-23fcbfba4aca',
 'a68372ae-2185-4291-a6b6-707c60ba480c',
 'a65af5a6-4044-4a29-8f26-3b1983817996',
 '7da70330-00f4-4e15-bb8a-55473522a1af',
 '084fed68-b3a5-4986-94b9-07be859cba84',
 '2d4dc6b9-7513-4207-bcf6-53245de182ff',
 'df9b0fb1-2a8b-42fb-acb9-0a505f58ffc3',
 '550961c3-ea9a-423c-b742-2bdaaa049524',
 '6a103059-a13e-4dcd-a40a-c8b00b4233af',
 '7eba875b-737b-4fce-a2d1-e80fece24c1f',
 '7658006a-70a5-42a8-aea3-e2b978164fd9',
 '4c7bb4e1-235f-4081-a629-f54f62c0ac3e',
 '81c7825e-a15c-4a7b-8b7e-b82c581a2eac',
 '0194c297-38c9-4cd0-a0dd-9448fd3b07fa',
 '0d491822-59c8-4135-ae4e-9e472e214e7b',
 '4bd3a879-8c50-4678-839a-2cfd7025b457',
 '664e1224-94db-4687-b20c-ff21a60b6619',
 '964332b7-c704-4224-8f0c-2e36c584dc17',
 'da552c62-ca24-4f95-9862-d1a0c74fcb8b',
 '535f66ae-b5ac-4a3b-8955-4d40fd569fb1',
 'e8f612e8-a826-45e1-93dd-de3b787ec0e8',
 '3a9fd55b-98c7-4987-bd6d-f4ef2f555d1d',
 'ed82e38c-b7df-4771-9607-483e4e9a6cb3',
 'c1fa1448-01f9-

In [93]:
# Defining Retriver
retriver=vector_store.as_retriever(search_kwargs={"k":5,"lambda_mult":1},search_type="mmr")

In [65]:
query1="What Happenened in Srinagar Airport"
query1Res=retriver.invoke(query1)
print(query1Res)

[Document(id='8f1e2ecf-01d1-43f8-83b7-9d4bb16458f9', metadata={}, page_content='it being looked at as something more? Well, uh as you are aware that Srinagar Airport is a very high security airport. So,'), Document(id='4c7bb4e1-235f-4081-a629-f54f62c0ac3e', metadata={}, page_content='it being looked at as something more? Well, uh as you are aware that Srinagar Airport is a very high security airport. So,'), Document(id='4bdcba32-63d2-4bff-8041-521aef72b572', metadata={}, page_content="any other airport, that's a different story. But in Kashmir, uh every such incident uh you can't take that on face value.")]


In [45]:
# Defining Special Retriver
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [54]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [66]:
mqr=MultiQueryRetriever.from_llm(
    llm=llm,
    retriever=retriver
)

In [67]:
ccr=ContextualCompressionRetriever(
    base_compressor=LLMChainExtractor.from_llm(llm),
    base_retriever=mqr
)

In [68]:
query2="What are we talking about here?"
query2res=ccr.invoke(query2)

In [69]:
print(query2res)

[Document(metadata={}, page_content="uh you can't take that on face value. You can't take that as a mistake. You have to go deep, investigate. And that's what"), Document(metadata={}, page_content="Today's Mufti Farid. Mufti, is this uh being investigated as a matter of mere confusion because uh many people are not aware"), Document(metadata={}, page_content='and uh considering the present security situation all around, uh the war-like situation, there are many factors that have'), Document(metadata={}, page_content='and uh considering the present security situation all around, uh the war-like situation, there are many factors that have')]


In [52]:
# Now The Argumentation Part 
from langchain_core.prompts import PromptTemplate

prompt=PromptTemplate(
    template="""
      You are a Calm Intellectual Person and you will tell me what is the Significane of the following content 
      use only the provided context 
      Context:- {context}
      query:- {query}
    """,
    input_variables=["context","query"]
)

In [ ]:
query3="What Happens at The Sri Nagar Airport?"
query3res=mqr.invoke(query3)

In [71]:
print(query3res)

[Document(id='8f1e2ecf-01d1-43f8-83b7-9d4bb16458f9', metadata={}, page_content='it being looked at as something more? Well, uh as you are aware that Srinagar Airport is a very high security airport. So,'), Document(id='4c7bb4e1-235f-4081-a629-f54f62c0ac3e', metadata={}, page_content='it being looked at as something more? Well, uh as you are aware that Srinagar Airport is a very high security airport. So,'), Document(id='4bdcba32-63d2-4bff-8041-521aef72b572', metadata={}, page_content="any other airport, that's a different story. But in Kashmir, uh every such incident uh you can't take that on face value.")]


In [72]:
query3llm = prompt.format(
    context=query3res,
    query=query3
)

llm_output=llm.invoke(query3llm)

In [74]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()

In [75]:
print(parser.invoke(llm_output))

Based on the provided context, the significance of the content is that Srinagar Airport is a "very high security airport." This designation suggests that incidents occurring there may be viewed with a higher degree of scrutiny or as potentially more significant than at other airports.


In [82]:
# Defining The Chains
from langchain_core.runnables import RunnableParallel,RunnableLambda,RunnablePassthrough

In [77]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [94]:
parallel_chain=RunnableParallel({
    'context': retriver | RunnableLambda(format_docs),
    'query': RunnablePassthrough()
})

In [95]:
chain=parallel_chain | prompt | llm | parser

In [96]:
chain_res=chain.invoke("What Happened at Sri Nagar Airport")

In [97]:
print(chain_res)

Based on the provided context, two US nationals were detained by Srinagar Airport security. The significance of this event in Kashmir is that "every such incident you can't take that on face value," implying it is viewed as something more due to Srinagar Airport being a "very high security airport."
